# 06 — Cox Proportional Hazards — FINAL

Chạy cả Day-0 và Day-7 theo `evaluation.modes`.

- event = resolution
- HR > 1: resolution hazard cao hơn → xu hướng resolve nhanh hơn
- HR < 1: resolution hazard thấp hơn → xu hướng tồn đọng lâu hơn
- Không diễn giải Hazard Ratio thành quan hệ nhân quả

In [1]:
from pathlib import Path
import sys, yaml, json, time, gc, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

with open(ROOT / "configs" / "experiment.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

SEED = int(CFG["random_seed"])
MODES = list(CFG.get("evaluation", {}).get("modes", ["day0"]))

print("ROOT =", ROOT)
print("random_seed =", SEED)
print("modes =", MODES)

ROOT = d:\VNUK\Eureka 2026\Eureka_2026
random_seed = 42
modes = ['day0', 'day7']


In [2]:
from src.features import feature_spec, sample_training_rows
from src.models import fit_cox, save_artifact
from src.evaluation import evaluate_survival_model
from src.splitting import make_project_split

model_dir = ROOT / "results" / "models"
table_dir = ROOT / "results" / "tables"
model_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

all_metrics = []

In [3]:
for mode in MODES:
    print("\n" + "=" * 100)
    print("COX FINAL MODE:", mode)
    print("=" * 100)

    data_path = ROOT / "data" / "processed" / f"model_{mode}.parquet"
    if not data_path.exists():
        raise FileNotFoundError(
            f"Không thấy {data_path}. Hãy chạy lại 04_feature_engineering.ipynb trước."
        )

    data = pd.read_parquet(data_path)
    train, val, test = make_project_split(data, random_state=SEED)

    print("Train projects:", sorted(train["project_uid"].unique()))
    print("Validation projects:", sorted(val["project_uid"].unique()))
    print("Test projects:", sorted(test["project_uid"].unique()))

    strict = bool(CFG["features"]["strict_no_leakage"])
    landmark = int(CFG["features"]["landmark_days"])

    numeric_cols, categorical_cols = feature_spec(
        mode,
        strict_no_leakage=strict,
        landmark_days=landmark,
    )

    max_rows = CFG["models"].get("max_train_rows")
    train_fit = sample_training_rows(
        train,
        max_rows=max_rows,
        random_state=SEED,
    )

    print("Train rows used:", len(train_fit), "/", len(train))

    alpha = float(CFG["models"]["cox"]["alpha"])
    started = time.time()

    cox_artifact = fit_cox(
        train_df=train_fit,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        alpha=alpha,
    )

    coef_arr = np.asarray(cox_artifact["model"].coef_)
    if not np.all(np.isfinite(coef_arr)):
        raise RuntimeError(
            f"Cox {mode}: non-finite coefficients. Tăng alpha trong experiment.yaml."
        )

    eval_cfg = CFG["evaluation"]

    metrics_val = evaluate_survival_model(
        cox_artifact,
        train_df=train_fit,
        test_df=val,
        horizons_days=eval_cfg["horizons_days"],
        max_ibs_rows=eval_cfg["max_ibs_rows"],
        survival_batch_size=eval_cfg["survival_batch_size"],
        random_state=SEED,
    )

    metrics_test = evaluate_survival_model(
        cox_artifact,
        train_df=train_fit,
        test_df=test,
        horizons_days=eval_cfg["horizons_days"],
        max_ibs_rows=eval_cfg["max_ibs_rows"],
        survival_batch_size=eval_cfg["survival_batch_size"],
        random_state=SEED + 1,
    )

    elapsed = time.time() - started

    save_artifact(cox_artifact, model_dir / f"cox_{mode}.joblib")

    metrics_df = pd.DataFrame([
        {
            "model": "CoxPH",
            "mode": mode,
            "split": "validation",
            "train_rows": len(train_fit),
            "wall_seconds": elapsed,
            **metrics_val,
        },
        {
            "model": "CoxPH",
            "mode": mode,
            "split": "test",
            "train_rows": len(train_fit),
            "wall_seconds": elapsed,
            **metrics_test,
        },
    ])
    metrics_df.to_csv(table_dir / f"cox_metrics_{mode}.csv", index=False)
    all_metrics.append(metrics_df)

    # Lưu project-disjoint split
    split_frames = []
    for split_name, split_df in [
        ("train", train),
        ("validation", val),
        ("test", test),
    ]:
        x = split_df[
            ["repository", "project", "issue_key", "project_uid"]
        ].copy()
        x["split"] = split_name
        split_frames.append(x)

    pd.concat(split_frames, ignore_index=True).to_csv(
        table_dir / f"split_{mode}.csv",
        index=False,
    )

    # Hazard ratios
    coef = np.asarray(cox_artifact["model"].coef_, dtype=float)
    hr = pd.DataFrame({
        "feature": cox_artifact["feature_names"],
        "coef": coef,
        "hazard_ratio": np.exp(np.clip(coef, -50, 50)),
    })
    hr["abs_log_hr"] = np.abs(hr["coef"])
    hr = hr.sort_values("abs_log_hr", ascending=False)
    hr.to_csv(table_dir / f"cox_hazard_ratios_{mode}.csv", index=False)

    display(metrics_df)
    display(hr.head(20))

    del data, train, val, test, train_fit, cox_artifact
    gc.collect()

cox_all = pd.concat(all_metrics, ignore_index=True)
cox_all.to_csv(table_dir / "cox_metrics_ALL_MODES.csv", index=False)
cox_all


COX FINAL MODE: day0
Train projects: ['Apache::HIVE', 'Jira::CONFSERVER', 'Jira::JRACLOUD', 'Jira::JRASERVER', 'MariaDB::MDEV', 'Mojang::MC', 'Mojang::MCPE', 'Sonatype::OSSRH']
Validation projects: ['Apache::FLEX', 'Sakai::SAK']
Test projects: ['MongoDB::SERVER', 'Qt::QTBUG']
Train rows used: 100000 / 600802


,model,mode,split,train_rows,wall_seconds,n_test,events_test,harrell_c,n_ipcw_eval,ipcw_c,auc_30,auc_60,auc_90,mean_dynamic_auc,ibs_rows,ibs,km_ibs,ibs_gain_vs_km
0,CoxPH,day0,validation,100000,580.72391,78741,70650,0.570553,78741,0.575278,0.600919,0.600744,0.603605,0.601089,2000,0.185307,0.180345,-0.004962
1,CoxPH,day0,test,100000,580.72391,156100,130492,0.531058,156100,0.531182,0.542912,0.551793,0.560501,0.545595,2000,0.183790,0.183822,0.000032


,feature,coef,hazard_ratio,abs_log_hr
25,cat__initial_issue_type_Problem,1.735027,5.669082,1.735027
33,cat__initial_issue_type_Suggestion,-1.644227,0.193162,1.644227
40,cat__initial_issue_type_Wish,-1.465767,0.230901,1.465767
27,cat__initial_issue_type_Publishing Support,1.242170,3.463120,1.242170
24,cat__initial_issue_type_New Project,1.219898,3.386841,1.219898
3,cat__initial_priority_,1.079193,2.942304,1.079193
26,cat__initial_issue_type_Public Security Vulner...,1.047505,2.850531,1.047505
20,cat__initial_issue_type_Feedback,-1.010099,0.364183,1.010099
30,cat__initial_issue_type_Story,-0.697234,0.497961,0.697234
14,cat__initial_priority_Unknown,0.667757,1.949859,0.667757



COX FINAL MODE: day7
Train projects: ['Apache::HIVE', 'Jira::CONFSERVER', 'Jira::JRACLOUD', 'Jira::JRASERVER', 'MariaDB::MDEV', 'Mojang::MC', 'Mojang::MCPE', 'Sonatype::OSSRH']
Validation projects: ['Apache::FLEX', 'Sakai::SAK']
Test projects: ['MongoDB::SERVER', 'Qt::QTBUG']
Train rows used: 100000 / 250679


,model,mode,split,train_rows,wall_seconds,n_test,events_test,harrell_c,n_ipcw_eval,ipcw_c,auc_30,auc_60,auc_90,mean_dynamic_auc,ibs_rows,ibs,km_ibs,ibs_gain_vs_km
0,CoxPH,day7,validation,100000,278.150009,55295,47222,0.523030,55295,0.544791,0.537024,0.543152,0.546091,0.539682,2000,0.230178,0.211349,-0.018829
1,CoxPH,day7,test,100000,278.150009,114055,88564,0.562469,114055,0.559500,0.600506,0.586518,0.585283,0.595193,2000,0.219886,0.206897,-0.012989


,feature,coef,hazard_ratio,abs_log_hr
36,cat__initial_issue_type_Publishing Support,1.763267,5.831459,1.763267
35,cat__initial_issue_type_Public Security Vulner...,1.671019,5.317583,1.671019
22,cat__initial_issue_type_Atlassian Task,1.625672,5.081832,1.625672
11,cat__initial_priority_Complex Fast-Track,1.370679,3.938022,1.370679
49,cat__initial_issue_type_Wish,-1.302264,0.271916,1.302264
42,cat__initial_issue_type_Suggestion,-1.235195,0.290778,1.235195
29,cat__initial_issue_type_Feedback,-0.965987,0.380607,0.965987
34,cat__initial_issue_type_Problem,0.893211,2.442961,0.893211
48,cat__initial_issue_type_User Story,0.733541,2.082442,0.733541
16,cat__initial_priority_Major,-0.628874,0.533192,0.628874


,model,mode,split,train_rows,wall_seconds,n_test,events_test,harrell_c,n_ipcw_eval,ipcw_c,auc_30,auc_60,auc_90,mean_dynamic_auc,ibs_rows,ibs,km_ibs,ibs_gain_vs_km
0,CoxPH,day0,validation,100000,580.723910,78741,70650,0.570553,78741,0.575278,0.600919,0.600744,0.603605,0.601089,2000,0.185307,0.180345,-0.004962
1,CoxPH,day0,test,100000,580.723910,156100,130492,0.531058,156100,0.531182,0.542912,0.551793,0.560501,0.545595,2000,0.183790,0.183822,0.000032
2,CoxPH,day7,validation,100000,278.150009,55295,47222,0.523030,55295,0.544791,0.537024,0.543152,0.546091,0.539682,2000,0.230178,0.211349,-0.018829
3,CoxPH,day7,test,100000,278.150009,114055,88564,0.562469,114055,0.559500,0.600506,0.586518,0.585283,0.595193,2000,0.219886,0.206897,-0.012989


## PH assumption

Khi viết báo cáo cuối, nên bổ sung proportional-hazards diagnostic (Schoenfeld) trên một diagnostic sample đủ lớn. Phần core FINAL ở đây ưu tiên mô hình + metrics ổn định trước.